In [1]:
#!pip install torchvision
#!pip install facenet_pytorch

import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
# from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
# from vivit import ViViTBackbone
import gc
from VideoSwinTransformer import SwinTransformer3D
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="r2plus1d+VST + AST fusion after transformer blocks"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()


In [2]:

logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:3' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-23 17:03:20,724]::4281476926::DEBUG::=============r2plus1d+VST + AST fusion after transformer blocks+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:3


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)

1


In [4]:
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

In [5]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
print(1)

1


In [6]:
train_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/train_set.dat","train")
print(1)
validation_set_data=getdata("/data1/qx/mean_based_arg/udiva/multimodal_fusion/data/speaking_only/valid_set.dat","valid")
print(1)

1
1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,audiodata,fullshot,tagdata,transform=None):
        self.audiodata=audiodata
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        audiodata = self.audiodata[idx]
        audiodata = torch.FloatTensor(audiodata)
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return audiodata,fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20
lr = 3e-05
epochs = 120
print(1)

1


In [10]:
train_set_data = ChalearnDataset(audiodata=train_set_data[0],fullshot = train_set_data[1],tagdata=train_set_data[2],transform=transform)
val_set_data = ChalearnDataset(audiodata=validation_set_data[0],fullshot = validation_set_data[1],tagdata=validation_set_data[2],transform=transform)
train_dataloader = DataLoader(dataset=train_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
val_dataloader = DataLoader(dataset=val_set_data, batch_size=batchsz, shuffle=True, num_workers=num_workerssz)
max_value=0
print(1)

1


In [11]:
model=SwinTransformer3D()#num_frames=frame_nums,batchsz = batchsz
model.to(device)
criterion = torch.nn.MSELoss().to(device)  # 손실함수
optimizer = torch.optim.AdamW(model.parameters(), lr=lr) # 옵티마이저

if test_flage==True:
    checkpoint=torch.load(save_model_file_path.format('model',start_epoch,'pth'), map_location=device)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    # criterion.load_state_dict(checkpoint["loss"])
train_avg_loss0=[]
val_avg_loss0=[]
print(1)

---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299
1


In [12]:
from torchinfo import summary

summary(model, input_size = ((4, 3,15, 224, 224),(4,24,3000,1)), col_names = ['input_size','output_size','num_params'])

Layer (type:depth-idx)                             Input Shape               Output Shape              Param #
SwinTransformer3D                                  [4, 3, 15, 224, 224]      [4, 5]                    --
├─ASTModel: 1-1                                    [4, 3000, 24]             [4, 768]                  885,120
│    └─DistilledVisionTransformer: 2-1             --                        --                        1,770,704
│    │    └─PatchEmbed: 3-1                        [4, 1, 24, 3000]          [4, 299, 768]             197,376
│    │    └─Dropout: 3-2                           [4, 301, 768]             [4, 301, 768]             --
│    │    └─ModuleList: 3-3                        --                        --                        85,054,464
│    │    └─LayerNorm: 3-4                         [4, 301, 768]             [4, 301, 768]             1,536
│    └─Sequential: 2-2                             [4, 768]                  [4, 768]                  --
│    │    └─L

In [13]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [14]:


# ============================================================
# Reproducibility / Multi-seed / Early-stopping settings
# ============================================================
import random
import numpy as np
import torch

SEEDS = [42, 123, 777, 1004, 2024]
patience = 20

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# ============================================================
# UDIVA Multi-Seed Training — DIRECT MSE ONLY
# Overall and O/C/E/A/N metrics are raw MSE (lower is better).
# Never use 1-MSE or 1-MAE.
# ============================================================

seed_results = []
global_best_mse = float("inf")
global_best_seed = None
global_best_epoch = None
best_model_path = "./save_swintransformer_folder/model_best_multiseed_direct_mse.pth"

with torch.cuda.device(3):
    for seed in SEEDS:
        print("\n" + "=" * 80)
        print(f"START SEED {seed} | DIRECT MSE")
        print("=" * 80)

        set_seed(seed)

        generator = torch.Generator()
        generator.manual_seed(seed)

        train_dataloader = DataLoader(
            dataset=train_set_data,
            batch_size=batchsz,
            shuffle=True,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )
        val_dataloader = DataLoader(
            dataset=val_set_data,
            batch_size=batchsz,
            shuffle=False,
            num_workers=num_workerssz,
            worker_init_fn=seed_worker,
            generator=generator
        )

        # Reinitialize model and optimizer for every seed.
        model = SwinTransformer3D()
        model.to(device)
        criterion = torch.nn.MSELoss().to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

        seed_best_mse = float("inf")
        seed_best_epoch = -1
        no_improve_count = 0

        for i in range(start_epoch, epochs):
            model.train()
            train_avg_loss = 0

            for audio,fullshot, big_five_data in tqdm(
                train_dataloader,
                desc=f"Seed {seed} | Train Epoch {i+1}/{epochs}"
            ):
                fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
                big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
                audio = audio.to(device)
                optimizer.zero_grad()
                hypothesis = model(fullshot,audio)
                loss = criterion(hypothesis, big_five_data)
                loss.backward()
                optimizer.step()
                train_avg_loss += loss

            train_avg_loss /= len(train_dataloader)

            model.eval()
            val_avg_loss = 0
            val_avg_o_loss = 0
            val_avg_c_loss = 0
            val_avg_e_loss = 0
            val_avg_a_loss = 0
            val_avg_n_loss = 0

            with torch.no_grad():
                for audio,fullshot, big_five_data in tqdm(
                    val_dataloader,
                    desc=f"Seed {seed} | Valid Epoch {i+1}/{epochs}"
                ):
                    fullshot = fullshot.permute(0, 4, 1, 2, 3).to(device)
                    big_five_data = big_five_data.permute(0, 2, 1).squeeze().to(device)
                    audio = audio.to(device)

                    hypothesis = model(fullshot,audio)
                    val_avg_loss += criterion(hypothesis, big_five_data)

                    hypothesiso, hypothesisc, hypothesise, hypothesisa, hypothesisn = \
                        extract_ocean(hypothesis, len(fullshot))
                    big_five_datao, big_five_datac, big_five_datae, \
                        big_five_dataa, big_five_datan = \
                        extract_ocean(big_five_data, len(fullshot))

                    val_avg_o_loss += criterion(hypothesiso, big_five_datao)
                    val_avg_c_loss += criterion(hypothesisc, big_five_datac)
                    val_avg_e_loss += criterion(hypothesise, big_five_datae)
                    val_avg_a_loss += criterion(hypothesisa, big_five_dataa)
                    val_avg_n_loss += criterion(hypothesisn, big_five_datan)

            val_avg_loss /= len(val_dataloader)
            val_avg_o_loss /= len(val_dataloader)
            val_avg_c_loss /= len(val_dataloader)
            val_avg_e_loss /= len(val_dataloader)
            val_avg_a_loss /= len(val_dataloader)
            val_avg_n_loss /= len(val_dataloader)

            current_val_mse = val_avg_loss.item()
            o_mse = val_avg_o_loss.item()
            c_mse = val_avg_c_loss.item()
            e_mse = val_avg_e_loss.item()
            a_mse = val_avg_a_loss.item()
            n_mse = val_avg_n_loss.item()

            if current_val_mse < seed_best_mse:
                seed_best_mse = current_val_mse
                seed_best_epoch = i + 1
                no_improve_count = 0
            else:
                no_improve_count += 1

            # Global checkpoint: lowest overall validation MSE among all seeds.
            if current_val_mse < global_best_mse:
                global_best_mse = current_val_mse
                global_best_seed = seed
                global_best_epoch = i + 1

                torch.save({
                    "seed": seed,
                    "epoch": i + 1,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "mse": current_val_mse,
                    "o_mse": o_mse,
                    "c_mse": c_mse,
                    "e_mse": e_mse,
                    "a_mse": a_mse,
                    "n_mse": n_mse,
                }, best_model_path)

            # EVERY EPOCH: direct/raw MSE only.
            print(
                f"Seed {seed} | Epoch {i+1} | "
                f"Val MSE = {current_val_mse:.6f} | "
                f"Best MSE = {seed_best_mse:.6f} | "
                f"Best Epoch = {seed_best_epoch} | "
                f"Patience = {no_improve_count}/{patience}"
            )
            print(
                f"O-MSE = {o_mse:.6f} | C-MSE = {c_mse:.6f} | "
                f"E-MSE = {e_mse:.6f} | A-MSE = {a_mse:.6f} | "
                f"N-MSE = {n_mse:.6f}"
            )

            if no_improve_count >= patience:
                print(
                    f"EARLY STOPPING | Seed {seed} | Epoch {i+1} | "
                    f"Best MSE = {seed_best_mse:.6f} | "
                    f"Best Epoch = {seed_best_epoch} | "
                    f"Patience = {no_improve_count}/{patience}"
                )
                break

        seed_results.append({
            "seed": seed,
            "best_mse": seed_best_mse,
            "best_epoch": seed_best_epoch
        })

        print(
            f"SEED {seed} FINISHED | Best MSE = {seed_best_mse:.6f} | "
            f"Best Epoch = {seed_best_epoch}"
        )

        del model, optimizer, criterion, train_dataloader, val_dataloader, generator
        gc.collect()
        torch.cuda.empty_cache()

# ============================================================
# Multi-Seed DIRECT MSE Summary
# ============================================================
all_mse = np.array([r["best_mse"] for r in seed_results], dtype=np.float64)
mean_mse = all_mse.mean()
std_mse = all_mse.std(ddof=1) if len(all_mse) > 1 else 0.0

print("\n" + "=" * 80)
print("MULTI-SEED DIRECT MSE RESULTS")
print("=" * 80)
for r in seed_results:
    print(
        f"Seed {r['seed']} | Best MSE = {r['best_mse']:.6f} | "
        f"Best Epoch = {r['best_epoch']}"
    )
print("-" * 80)
print(f"Mean Best MSE     = {mean_mse:.6f}")
print(f"Std Best MSE      = {std_mse:.6f}")
print(f"Global Best MSE   = {global_best_mse:.6f}")
print(f"Global Best Seed  = {global_best_seed}")
print(f"Global Best Epoch = {global_best_epoch}")
print(f"Patience limit    = {patience}")
print(f"Saved checkpoint  = {best_model_path}")
print("=" * 80)



START SEED 42 | DIRECT MSE
---------------AST Model Summary---------------
ImageNet pretraining: True, AudioSet pretraining: True
frequncey stride=10, time stride=10
number of patches=299


Seed 42 | Train Epoch 1/120: 100%|██████████| 1091/1091 [04:34<00:00,  3.97it/s]
Seed 42 | Valid Epoch 1/120: 100%|██████████| 191/191 [00:18<00:00, 10.48it/s]


Seed 42 | Epoch 1 | Val MSE = 1.436615 | Best MSE = 1.436615 | Best Epoch = 1 | Patience = 0/20
O-MSE = 1.522498 | C-MSE = 1.360296 | E-MSE = 1.073027 | A-MSE = 1.533095 | N-MSE = 1.694155


Seed 42 | Train Epoch 2/120: 100%|██████████| 1091/1091 [04:35<00:00,  3.96it/s]
Seed 42 | Valid Epoch 2/120: 100%|██████████| 191/191 [00:18<00:00, 10.38it/s]


Seed 42 | Epoch 2 | Val MSE = 1.500192 | Best MSE = 1.436615 | Best Epoch = 1 | Patience = 1/20
O-MSE = 1.262408 | C-MSE = 1.711630 | E-MSE = 1.670382 | A-MSE = 1.425320 | N-MSE = 1.431214


...

Seed 2024 | Epoch 19 | Val MSE = 1.429637 | Best MSE = 1.325034 | Best Epoch = 1 | Patience = 18/20
O-MSE = 1.110202 | C-MSE = 1.391731 | E-MSE = 1.321169 | A-MSE = 1.762810 | N-MSE = 1.562273


Seed 2024 | Train Epoch 20/120: 100%|██████████| 1091/1091 [04:37<00:00,  3.94it/s]
Seed 2024 | Valid Epoch 20/120: 100%|██████████| 191/191 [00:18<00:00, 10.44it/s]


Seed 2024 | Epoch 20 | Val MSE = 1.489467 | Best MSE = 1.325034 | Best Epoch = 1 | Patience = 19/20
O-MSE = 1.163999 | C-MSE = 1.432157 | E-MSE = 1.506819 | A-MSE = 1.801715 | N-MSE = 1.542645


Seed 2024 | Train Epoch 21/120: 100%|██████████| 1091/1091 [04:37<00:00,  3.94it/s]
Seed 2024 | Valid Epoch 21/120: 100%|██████████| 191/191 [00:17<00:00, 10.64it/s]

Seed 2024 | Epoch 21 | Val MSE = 1.445210 | Best MSE = 1.325034 | Best Epoch = 1 | Patience = 20/20
O-MSE = 1.187999 | C-MSE = 1.361093 | E-MSE = 1.407737 | A-MSE = 1.759408 | N-MSE = 1.509810
EARLY STOPPING | Seed 2024 | Epoch 21 | Best MSE = 1.325034 | Best Epoch = 1 | Patience = 20/20
SEED 2024 FINISHED | Best MSE = 1.325034 | Best Epoch = 1

MULTI-SEED DIRECT MSE RESULTS
Seed 42 | Best MSE = 1.289924 | Best Epoch = 10
Seed 123 | Best MSE = 1.316029 | Best Epoch = 8
Seed 777 | Best MSE = 1.260152 | Best Epoch = 15
Seed 1004 | Best MSE = 1.348863 | Best Epoch = 2
Seed 2024 | Best MSE = 1.325034 | Best Epoch = 1
--------------------------------------------------------------------------------
Mean Best MSE     = 1.308000
Std Best MSE      = 0.034061
Global Best MSE   = 1.260152
Global Best Seed  = 777
Global Best Epoch = 15
Patience limit    = 20
Saved checkpoint  = ./save_swintransformer_folder/model_best_multiseed_direct_mse.pth


In [15]:
# from torchinfo import summary

# summary(model, input_size = (4,15,4,3,224,224))